## Writing Complex Query

In [2]:
import os
from dotenv import load_dotenv

import pandas as pd
import sqlalchemy

In [3]:
load_dotenv()

db_host = os.environ.get("db_host")
db_user = os.environ.get("db_user")
db_password = os.environ.get("db_password")

In [4]:
engine = sqlalchemy.create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/sql_store")

In [5]:
pd.read_sql("SHOW TABLES", con= engine)

,Tables_in_sql_store
0,customers
1,order_item_notes
2,order_items
3,order_statuses
4,orders
5,products
6,shippers


## Subqueries

In [6]:
query = """
select *
from products 
where unit_price > (
    select unit_price
    from products
    where product_id = 3
)
"""

pd.read_sql(query, con= engine)

,product_id,name,quantity_in_stock,unit_price
0,2,"Pork - Bacon,back Peameal",49,4.65
1,4,"Brocolinni - Gaylan, Chinese",90,4.53


## IN Operator

In [7]:
query = """
select *
from products 
where product_id not in (
    select distinct product_id
    from order_items
)
"""

pd.read_sql(query, con= engine)

,product_id,name,quantity_in_stock,unit_price
0,7,Sweet Pea Sprouts,98,3.29


## Subqueries vs Joins
In general, JOINs are often better for readability and performance when you’re working with related tables, while subqueries are useful for more complex or nested logic.

In [8]:
query = """
select 
	customer_id,
    first_name,
    last_name
from customers
where customer_id in (
	select o.customer_id
	from order_items oi
    join orders o using(order_id)
	where product_id = 3
)
"""

pd.read_sql(query, con= engine)

,customer_id,first_name,last_name
0,8,Thacher,Naseby
1,2,Ines,Brushfield
2,10,Levy,Mynett


this code is same as above:

In [9]:
query = """
select distinct 
	c.customer_id,
    c.first_name,
    c.last_name
from customers c
join orders o using (customer_id)
join order_items oi using (order_id)
where oi.product_id = 3
"""

pd.read_sql(query, con= engine)

,customer_id,first_name,last_name
0,8,Thacher,Naseby
1,2,Ines,Brushfield
2,10,Levy,Mynett


## ALL Keyword

In [12]:
query = """
select *
from sql_invoicing.invoices
where invoice_total > all (
    select invoice_total
    from sql_invoicing.invoices 
    where client_id = 3
)
"""

pd.read_sql(query, con= engine)

,invoice_id,number,client_id,invoice_total,payment_total,invoice_date,due_date,payment_date
0,2,03-898-6735,5,175.32,8.18,2019-06-11,2019-07-01,2019-02-12
1,5,87-052-3121,5,169.36,0.00,2019-07-18,2019-08-07,None
2,8,78-145-1093,1,189.12,0.00,2019-05-20,2019-06-09,None
3,9,77-593-0081,5,172.17,0.00,2019-07-09,2019-07-29,None
4,18,52-269-9803,5,180.17,42.77,2019-05-23,2019-06-12,2019-01-08


## ANY Keyword

In [13]:
query = """
select *
from sql_invoicing.invoices
where invoice_total > any (
    select invoice_total
    from sql_invoicing.invoices 
    where client_id = 3
)
"""

pd.read_sql(query, con= engine)

,invoice_id,number,client_id,invoice_total,payment_total,invoice_date,due_date,payment_date
0,2,03-898-6735,5,175.32,8.18,2019-06-11,2019-07-01,2019-02-12
1,3,20-228-0335,5,147.99,0.00,2019-07-31,2019-08-20,None
2,4,56-934-0748,3,152.21,0.00,2019-03-08,2019-03-28,None
3,5,87-052-3121,5,169.36,0.00,2019-07-18,2019-08-07,None
4,6,75-587-6626,1,157.78,74.55,2019-01-29,2019-02-18,2019-01-03
5,7,68-093-9863,3,133.87,0.00,2019-09-04,2019-09-24,None
6,8,78-145-1093,1,189.12,0.00,2019-05-20,2019-06-09,None
7,9,77-593-0081,5,172.17,0.00,2019-07-09,2019-07-29,None
8,10,48-266-1517,1,159.50,0.00,2019-06-30,2019-07-20,None
9,13,41-666-1035,5,135.01,87.44,2019-06-25,2019-07-15,2019-01-26


## Correlated Subqueries
A correlated subquery is a subquery that depends on the outer query for each row, meaning it is executed repeatedly once for each row of the main query. It often references columns from the outer query to filter or compare data dynamically.

In [17]:
query = """
select *
from sql_invoicing.invoices i
where invoice_total > (
	select 
		avg(invoice_total)
	from sql_invoicing.invoices
    where client_id = i.client_id
)
"""

pd.read_sql(query, con= engine)

,invoice_id,number,client_id,invoice_total,payment_total,invoice_date,due_date,payment_date
0,2,03-898-6735,5,175.32,8.18,2019-06-11,2019-07-01,2019-02-12
1,4,56-934-0748,3,152.21,0.00,2019-03-08,2019-03-28,None
2,5,87-052-3121,5,169.36,0.00,2019-07-18,2019-08-07,None
3,8,78-145-1093,1,189.12,0.00,2019-05-20,2019-06-09,None
4,9,77-593-0081,5,172.17,0.00,2019-07-09,2019-07-29,None
5,15,55-105-9605,3,167.29,80.31,2019-11-25,2019-12-15,2019-01-15
6,16,10-451-8824,1,162.02,0.00,2019-03-30,2019-04-19,None
7,18,52-269-9803,5,180.17,42.77,2019-05-23,2019-06-12,2019-01-08


## EXISTS

In [22]:
query = """
select *
from sql_invoicing.clients c 
where exists (
	select 1
    from sql_invoicing.invoices 
    where client_id = c.client_id
)
"""

pd.read_sql(query, con= engine)

,client_id,name,address,city,state,phone
0,1,Vinte,3 Nevada Parkway,Syracuse,NY,315-252-7305
1,2,Myworks,34267 Glendale Parkway,Huntington,WV,304-659-1170
2,3,Yadel,096 Pawling Parkway,San Francisco,CA,415-144-6037
3,5,Topiclounge,0863 Farmco Road,Portland,OR,971-888-9129


this code is same as above but it's better to use `EXISTS` instead of `IN` because of the performance:

In [23]:
query = """
select *
from sql_invoicing.clients c 
where client_id in (
	select distinct i.client_id
    from sql_invoicing.invoices i
    where i.client_id = c.client_id
)
"""

pd.read_sql(query, con= engine)

,client_id,name,address,city,state,phone
0,1,Vinte,3 Nevada Parkway,Syracuse,NY,315-252-7305
1,2,Myworks,34267 Glendale Parkway,Huntington,WV,304-659-1170
2,3,Yadel,096 Pawling Parkway,San Francisco,CA,415-144-6037
3,5,Topiclounge,0863 Farmco Road,Portland,OR,971-888-9129
